# Phase 3 — Bayesian Preference Learning · Summary Metrics
Notebook này đọc kết quả Phase 3 từ Google Drive và in toàn bộ metrics.


In [11]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import gdown
from pathlib import Path

# File IDs từ Google Drive links
REPORT_ID  = "1Kwoas6wOVe3p08SHKghsl1GvS3SHaiz8"   # bayesian_preference_report.json
RESULTS_ID = "170Rj4cSupcns8d7GOGQAvV2t-vsaW8nq"   # bayesian_preference_results.csv

REPORT_PATH  = Path("bayesian_preference_report.json")
RESULTS_PATH = Path("bayesian_preference_results.csv")

gdown.download(f"https://drive.google.com/uc?id={REPORT_ID}",  str(REPORT_PATH),  quiet=False)
gdown.download(f"https://drive.google.com/uc?id={RESULTS_ID}", str(RESULTS_PATH), quiet=False)

print("✅ Download xong:")
print(f"  {REPORT_PATH}  ({REPORT_PATH.stat().st_size:,} bytes)")
print(f"  {RESULTS_PATH} ({RESULTS_PATH.stat().st_size:,} bytes)")


Downloading...
From: https://drive.google.com/uc?id=1Kwoas6wOVe3p08SHKghsl1GvS3SHaiz8
To: /content/bayesian_preference_report.json
100%|██████████| 3.54k/3.54k [00:00<00:00, 9.42MB/s]
Downloading...
From: https://drive.google.com/uc?id=170Rj4cSupcns8d7GOGQAvV2t-vsaW8nq
To: /content/bayesian_preference_results.csv
100%|██████████| 7.22k/7.22k [00:00<00:00, 14.4MB/s]

✅ Download xong:
  bayesian_preference_report.json  (3,543 bytes)
  bayesian_preference_results.csv (7,218 bytes)


In [13]:
import json
import pandas as pd

with open(REPORT_PATH, encoding="utf-8") as f:
    report = json.load(f)

df    = pd.read_csv(RESULTS_PATH)
valid = df[df["valid_for_reward"] == True].copy()

print(f"Report loaded  — valid_locations: {report['valid_locations']}")
print(f"Results loaded — valid rows: {len(valid)}")


Report loaded  — valid_locations: 37
Results loaded — valid rows: 37


In [14]:
cfg = report["config"]
print("=" * 55)
print("CONFIG")
print("=" * 55)
print(f"  T* grid           : {cfg['t_grid']}")
print(f"  RH* grid          : {cfg['rh_grid']}")
print(f"  setpoint_sigma    : {cfg['setpoint_sigma']}")
print(f"  setpoint_weight   : {cfg['setpoint_weight']}")
print(f"  ac_on_weight      : {cfg['ac_on_weight']}")
print(f"  ac_off_weight     : {cfg['ac_off_weight']}")
print(f"  stable_weight     : {cfg['stable_weight']}")
print(f"  max_stable        : {cfg['max_stable_per_location']}")
print(f"  min_setpoint_obs  : {cfg['min_setpoint_observations']}")


CONFIG
  T* grid           : [23.0, 32.0, 0.25]
  RH* grid          : [55.0, 85.0, 1.0]
  setpoint_sigma    : 3.5
  setpoint_weight   : 0.25
  ac_on_weight      : 0.7
  ac_off_weight     : 0.4
  stable_weight     : 0.15
  max_stable        : 60
  min_setpoint_obs  : 5


In [15]:
print("=" * 55)
print("POPULATION METRICS")
print("=" * 55)
print(f"  Locations total   : {report['locations']}")
print(f"  Valid locations   : {report['valid_locations']}")
print(f"  Invalid locations : {report['invalid_locations']}")
print(f"  mean_T*           : {report['mean_T_star']:.4f} °C")
print(f"  mean_RH*          : {report['mean_RH_star']:.4f} %")
print(f"  mean_sigma_T      : {report['mean_sigma_T']:.4f} °C")
print(f"  mean_sigma_RH     : {report['mean_sigma_RH']:.4f} %")
print(f"  t_boundary_locs   : {report['t_boundary_locations']}")
print(f"  rh_boundary_locs  : {report['rh_boundary_locations']}")


POPULATION METRICS
  Locations total   : 47
  Valid locations   : 37
  Invalid locations : 10
  mean_T*           : 27.9527 °C
  mean_RH*          : 70.1351 %
  mean_sigma_T      : 0.2473 °C
  mean_sigma_RH     : 1.4452 %
  t_boundary_locs   : 0
  rh_boundary_locs  : 0


In [16]:
prior = report["prior"]
print("=" * 55)
print("PRIOR (Chinese dataset)")
print("=" * 55)
print(f"  rows_total        : {prior['rows_total']}")
print(f"  rows_used         : {prior['rows_used']}")
print(f"  prior_T* mean     : {prior['prior_t_mean']:.4f} °C")
print(f"  prior_T* sigma    : {prior['prior_t_sigma']:.4f} °C")
print(f"  prior_RH* mean    : {prior['prior_rh_mean']:.4f} %")
print(f"  prior_RH* sigma   : {prior['prior_rh_sigma']:.4f} %")

shift_t  = report["mean_T_star"]  - prior["prior_t_mean"]
shift_rh = report["mean_RH_star"] - prior["prior_rh_mean"]
print(f"\n  T*  shift : {prior['prior_t_mean']:.2f} → {report['mean_T_star']:.2f} °C  ({shift_t:+.2f} °C)")
print(f"  RH* shift : {prior['prior_rh_mean']:.2f} → {report['mean_RH_star']:.2f} %   ({shift_rh:+.2f} %)")

PRIOR (Chinese dataset)
  rows_total        : 2992
  rows_used         : 2474
  prior_T* mean     : 25.9081 °C
  prior_T* sigma    : 1.9346 °C
  prior_RH* mean    : 61.5105 %
  prior_RH* sigma   : 9.8323 %

  T*  shift : 25.91 → 27.95 °C  (+2.04 °C)
  RH* shift : 61.51 → 70.14 %   (+8.62 %)


In [17]:
print("=" * 55)
print("PER-APARTMENT STATS")
print("=" * 55)
print(f"  T* min            : {valid['T_star'].min():.2f} °C")
print(f"  T* max            : {valid['T_star'].max():.2f} °C")
print(f"  T* mean           : {valid['T_star'].mean():.4f} °C")
print(f"  T* std            : {valid['T_star'].std():.4f} °C")
print(f"  sigma_T min       : {valid['sigma_T'].min():.4f} °C")
print(f"  sigma_T max       : {valid['sigma_T'].max():.4f} °C")
print(f"  sigma_T mean      : {valid['sigma_T'].mean():.4f} °C")
print(f"  RH* min           : {valid['RH_star'].min():.2f} %")
print(f"  RH* max           : {valid['RH_star'].max():.2f} %")
print(f"  sigma_RH mean     : {valid['sigma_RH'].mean():.4f} %")

t_min_grid, t_max_grid   = cfg["t_grid"][0], cfg["t_grid"][1]
rh_min_grid, rh_max_grid = cfg["rh_grid"][0], cfg["rh_grid"][1]
t_bnd  = ((valid["T_star"]  == t_min_grid)  | (valid["T_star"]  == t_max_grid)).sum()
rh_bnd = ((valid["RH_star"] == rh_min_grid) | (valid["RH_star"] == rh_max_grid)).sum()
print(f"\n  T* boundary arts  : {t_bnd}  (at {t_min_grid} or {t_max_grid} °C)")
print(f"  RH* boundary arts : {rh_bnd}  (at {rh_min_grid} or {rh_max_grid} %)")


PER-APARTMENT STATS
  T* min            : 26.25 °C
  T* max            : 29.50 °C
  T* mean           : 27.9527 °C
  T* std            : 0.6893 °C
  sigma_T min       : 0.1473 °C
  sigma_T max       : 0.4296 °C
  sigma_T mean      : 0.2473 °C
  RH* min           : 64.00 %
  RH* max           : 79.00 %
  sigma_RH mean     : 1.4452 %

  T* boundary arts  : 0  (at 23.0 or 32.0 °C)
  RH* boundary arts : 0  (at 55.0 or 85.0 %)


In [18]:
import pandas as pd

print("=" * 55)
print("sigma_T DISTRIBUTION")
print("=" * 55)
bins = [0, 0.15, 0.20, 0.25, 0.30, 0.40, 1.0]
dist = pd.cut(valid["sigma_T"], bins=bins).value_counts().sort_index()
for bucket, count in dist.items():
    print(f"  {str(bucket):20s} : {count} apartments")

print("\n" + "=" * 55)
print("T* DISTRIBUTION")
print("=" * 55)
for t_val, count in valid["T_star"].value_counts().sort_index().items():
    print(f"  T* = {t_val:.2f} °C  : {count} apartments")


sigma_T DISTRIBUTION
  (0.0, 0.15]          : 1 apartments
  (0.15, 0.2]          : 12 apartments
  (0.2, 0.25]          : 10 apartments
  (0.25, 0.3]          : 5 apartments
  (0.3, 0.4]           : 7 apartments
  (0.4, 1.0]           : 2 apartments

T* DISTRIBUTION
  T* = 26.25 °C  : 1 apartments
  T* = 26.50 °C  : 1 apartments
  T* = 27.00 °C  : 1 apartments
  T* = 27.25 °C  : 3 apartments
  T* = 27.50 °C  : 5 apartments
  T* = 27.75 °C  : 6 apartments
  T* = 28.00 °C  : 8 apartments
  T* = 28.50 °C  : 7 apartments
  T* = 28.75 °C  : 1 apartments
  T* = 29.00 °C  : 3 apartments
  T* = 29.50 °C  : 1 apartments


In [19]:
m, s = valid["T_star"].mean(), valid["T_star"].std()

print("=" * 55)
print("OUTLIER APARTMENTS (T* > 2 std từ mean)")
print("=" * 55)
outliers = valid[abs(valid["T_star"] - m) > 2 * s]
if outliers.empty:
    print("  None")
else:
    for _, row in outliers.iterrows():
        print(f"  {row['apartment_id']:6s}  T*={row['T_star']:.2f}  sigma_T={row['sigma_T']:.3f}  obs={int(row['setpoint_observations'])}")

print("\n" + "=" * 55)
print("LOW CONFIDENCE (setpoint_observations < 20)")
print("=" * 55)
low = valid[valid["setpoint_observations"] < 20]
if low.empty:
    print("  None")
else:
    for _, row in low.iterrows():
        print(f"  {row['apartment_id']:6s}  T*={row['T_star']:.2f}  sigma_T={row['sigma_T']:.3f}  obs={int(row['setpoint_observations'])}")
        print(f"         → Phase 4 fallback: dùng population mean T*={m:.2f} °C")


OUTLIER APARTMENTS (T* > 2 std từ mean)
  AP16    T*=26.50  sigma_T=0.394  obs=35
  AP37    T*=29.50  sigma_T=0.241  obs=82
  AP4     T*=26.25  sigma_T=0.420  obs=19

LOW CONFIDENCE (setpoint_observations < 20)
  AP4     T*=26.25  sigma_T=0.420  obs=19
         → Phase 4 fallback: dùng population mean T*=27.95 °C
  AP6     T*=27.25  sigma_T=0.430  obs=14
         → Phase 4 fallback: dùng population mean T*=27.95 °C


In [20]:
print("=" * 55)
print("FULL VALID APARTMENT TABLE")
print("=" * 55)
cols = ["apartment_id","T_star","RH_star","sigma_T","sigma_RH",
        "setpoint_observations","stable_observations_used"]
display(valid[cols].reset_index(drop=True))
print("\n✅ Done.")


FULL VALID APARTMENT TABLE


,apartment_id,T_star,RH_star,sigma_T,sigma_RH,setpoint_observations,stable_observations_used
0,AP10,28.50,70.0,0.351410,1.950886,42,60
1,AP11,27.25,79.0,0.179041,1.112735,140,60
2,AP12,27.50,73.0,0.273590,1.649733,98,60
3,AP13,27.75,71.0,0.232250,1.349715,84,60
4,AP14,27.00,73.0,0.367485,2.094808,54,60
5,AP15,28.00,76.0,0.265725,1.555755,63,60
6,AP16,26.50,73.0,0.394292,2.218005,35,60
7,AP17,28.00,69.0,0.189797,1.139923,158,60
8,AP18,28.00,69.0,0.188478,1.121603,137,60
9,AP19,28.00,67.0,0.155912,0.939413,222,60



✅ Done.
